# CSC 2301 · Week 7 — Review and mini-project: a drilling campaign report

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/csc2301/notebooks/CSC2301_W7_review_mini_project.ipynb)

**IUGB · BSc Mining / Petroleum Engineering** · Instructor: Laurent Kouadio

Halfway through the course: time to put everything together. A reverse-circulation (RC) drilling campaign
of 8 holes has just been assayed (synthetic data). Your manager wants a **one-page report**: metres drilled,
mean grade and the best mineralised intercept of each hole.

Each question reuses one week of the course: reading files and types (W1), loops and decisions (W2),
functions (W3), strings and formatting (W4), dictionaries (W5), handling bad data (W6).
Work in small steps and check each result before moving on.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/csc2301/'
FILES = [('data/campaign_rc_assays.csv', 'campaign_rc_assays.csv'), ('tests/w7/q1.py', 'tests/q1.py'), ('tests/w7/q2.py', 'tests/q2.py'), ('tests/w7/q3.py', 'tests/q3.py'), ('tests/w7/q4.py', 'tests/q4.py'), ('tests/w7/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

## The brief

* `au_gpt` is the gold grade in g/t. The lab wrote `NS` (no sample) or the error code `-9.99` for some intervals:
  these are **not** grades.
* **Metres drilled** count every interval; **grades** use only valid intervals, **weighted by length**.
* An **intercept** is a run of **consecutive** valid intervals with grade ≥ 1.0 g/t. Geologists report the best one
  as, for example, *"12.0–18.0 m at 2.41 g/t"*. The best is the one with the most metal (grade × length).

Start by looking at the file.

In [ ]:
import csv

with open("campaign_rc_assays.csv", encoding="utf-8") as f:
    raw = list(csv.DictReader(f))
print(len(raw), "rows")
for r in raw[:6]:
    print(r)
print({r["au_gpt"] for r in raw if not r["au_gpt"].replace(".", "").isdigit()})

---
## Exercises

### Q1. Clean intervals

Build `intervals`: a list of dictionaries with keys `hole_id`, `from_m`, `to_m`, `au_gpt` (floats), only
for the rows with a **valid** grade (a number ≥ 0), in file order. Store their number in `n_valid`.
A `safe_float` like last week's helps.

In [ ]:
intervals = []
...
n_valid = len(intervals)
n_valid

In [ ]:
grader.check("q1")

### Q2. Metres drilled per hole

Build `metres_by_hole`: hole_id → total length drilled, using **all** rows of the file (`raw`), sampled or not.

In [ ]:
metres_by_hole = {}
...
metres_by_hole

In [ ]:
grader.check("q2")

### Q3. Mean grade per hole

Write `weighted_mean(rows)` that returns the length-weighted mean grade of a list of interval dictionaries,
then build `hole_grades`: hole_id → weighted mean of that hole's valid intervals.

In [ ]:
def weighted_mean(rows):
    ...

hole_grades = {}
...
hole_grades

In [ ]:
grader.check("q3")

### Q4. Best intercept

Write `best_intercept(rows, cutoff=1.0)` for the valid intervals of **one hole** (in depth order). It joins
consecutive intervals with grade ≥ `cutoff` into runs and returns the run with the most metal as a tuple
`(from_m, to_m, weighted_mean_grade)`, or `None` if no interval reaches the cut-off.

*Hint:* build the current run in a list; when an interval is below the cut-off (or the hole ends), the run is
finished: compare it with the best so far and start a new one. Reuse `weighted_mean`.

In [ ]:
def best_intercept(rows, cutoff=1.0):
    ...

best_intercept([r for r in intervals if r["hole_id"] == "RC-01"])

In [ ]:
grader.check("q4")

### Q5. The report

Write `campaign_report.txt` with one line per hole, **sorted by hole_id**, exactly in this format:

```
RC-01: 44.0 m drilled, mean 0.83 g/t, best 14.0-22.0 m at 2.35 g/t
```

(1 decimal for metres and depths, 2 for grades). For a hole without intercept, end the line with `best none`.

In [ ]:
lines = []
...
with open("campaign_report.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print(open("campaign_report.txt", encoding="utf-8").read())

In [ ]:
grader.check("q5")

### Going further (not checked)

* Change the cut-off to 0.5 g/t and to 2.0 g/t. How do the intercepts change? Why do companies like to
  choose the cut-off that gives the most impressive headline?
* Add the share of intervals not sampled (`NS`) to each line: is any hole less reliable than the others?

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()